# Octavio Vieyra Orozco

| Columna | Tipo de dato | ¿Qué representa? |
|---|---|---|
| `Administrative` | Entero (`int64`) | Cantidad de páginas administrativas visitadas, como las relacionadas con la cuenta del usuario. |
| `Administrative_Duration` | Decimal (`float64`) | Tiempo total dedicado a páginas administrativas, en segundos. |
| `Informational` | Entero (`int64`) | Cantidad de páginas informativas visitadas, como información sobre la tienda. |
| `Informational_Duration` | Decimal (`float64`) | Tiempo total dedicado a páginas informativas, en segundos. |
| `ProductRelated` | Entero (`int64`) | Cantidad de páginas relacionadas con productos visitadas. |
| `ProductRelated_Duration` | Decimal (`float64`) | Tiempo total dedicado a páginas de productos, en segundos. |
| `BounceRates` | Decimal (`float64`) | Tasa de rebote de las páginas visitadas: proporción de visitas que terminan sin otra interacción. |
| `ExitRates` | Decimal (`float64`) | Tasa de salida de las páginas visitadas: proporción de vistas en las que una página fue la última de la sesión. |
| `PageValues` | Decimal (`float64`) | Valor promedio atribuido a las páginas visitadas antes de completar una compra. |
| `SpecialDay` | Decimal (`float64`) | Indicador de cercanía a una fecha comercial especial. Toma valores entre 0 y 1. |
| `Month` | Texto (`object`), categórica | Mes en que ocurrió la sesión. |
| `OperatingSystems` | Entero (`int64`), categórica | Código del sistema operativo utilizado. Los números representan categorías. |
| `Browser` | Entero (`int64`), categórica | Código del navegador utilizado. Los números representan categorías. |
| `Region` | Entero (`int64`), categórica | Código de la región del visitante. |
| `TrafficType` | Entero (`int64`), categórica | Código del tipo o fuente de tráfico por el que llegó el visitante. |
| `VisitorType` | Texto (`object`), categórica | Tipo de visitante: nuevo, recurrente u otro. |
| `Weekend` | Booleano (`bool`) | Indica si la sesión ocurrió en fin de semana: `True` significa sí y `False`, no. |
| `Revenue` | Booleano (`bool`) | Indica si la sesión terminó en una compra: `True` significa sí y `False`, no. No representa un importe monetario. |

In [27]:
!pip -q install kagglehub

In [28]:
# librerias
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [29]:
import kagglehub
import pandas as pd
import os
from pathlib import Path

# Descargar el dataset dentro de Colab
path = Path(
    kagglehub.dataset_download(
        "imakash3011/online-shoppers-purchasing-intention-dataset"
    )
)

# Encontrar y leer el archivo CSV
csv_path = next(path.rglob("*.csv"))
df = pd.read_csv(csv_path)

Using Colab cache for faster access to the 'online-shoppers-purchasing-intention-dataset' dataset.


In [30]:
# Dimensiones del Dataset
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 12330
Columnas: 18


In [31]:
# Primera filas del Dataset
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.00,0,0.00,1,0.00,0.20,0.20,0.00,0.00,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.00,0,0.00,2,64.00,0.00,0.10,0.00,0.00,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.00,0,0.00,1,0.00,0.20,0.20,0.00,0.00,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.00,0,0.00,2,2.67,0.05,0.14,0.00,0.00,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.00,0,0.00,10,627.50,0.02,0.05,0.00,0.00,Feb,3,3,1,4,Returning_Visitor,True,False


In [32]:
# Información del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [33]:
# Calcula el número de valores únicos por columna y los ordena de forma descendente (de mayor a menor)
cardinalidad = df.nunique().sort_values(ascending=False)
# Muestra el resultado en pantalla
cardinalidad

,0
ProductRelated_Duration,9551
ExitRates,4777
Administrative_Duration,3335
PageValues,2704
BounceRates,1872
Informational_Duration,1258
ProductRelated,311
Administrative,27
TrafficType,20
Informational,17


In [34]:
# Describe variables cuantitativas
df.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,"12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00","12,330.00"
mean,2.32,80.82,0.50,34.47,31.73,"1,194.75",0.02,0.04,5.89,0.06,2.12,2.36,3.15,4.07
std,3.32,176.78,1.27,140.75,44.48,"1,913.67",0.05,0.05,18.57,0.20,0.91,1.72,2.40,4.03
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00
25%,0.00,0.00,0.00,0.00,7.00,184.14,0.00,0.01,0.00,0.00,2.00,2.00,1.00,2.00
50%,1.00,7.50,0.00,0.00,18.00,598.94,0.00,0.03,0.00,0.00,2.00,2.00,3.00,2.00
75%,4.00,93.26,0.00,0.00,38.00,"1,464.16",0.02,0.05,0.00,0.00,3.00,2.00,4.00,4.00
max,27.00,"3,398.75",24.00,"2,549.38",705.00,"63,973.52",0.20,0.20,361.76,1.00,8.00,13.00,9.00,20.00


In [35]:
# Describe variables cualitativas
df.describe(include="object")

,Month,VisitorType
count,12330,12330
unique,10,3
top,May,Returning_Visitor
freq,3364,10551


In [36]:
# Cardinalidad de los datos
# Calcula el número de valores únicos por columna y los ordena de forma descendente (de mayor a menor)
cardinalidad = df.nunique().sort_values(ascending=False)
# Muestra el resultado en pantalla
cardinalidad

,0
ProductRelated_Duration,9551
ExitRates,4777
Administrative_Duration,3335
PageValues,2704
BounceRates,1872
Informational_Duration,1258
ProductRelated,311
Administrative,27
TrafficType,20
Informational,17


## Comprensión de los Datos

### Duplicados
Dado que la cantidad de duplicados es demasiado baja, el eliminar estos no afectaria la callidad del dataset y el tener pocos datos repetidos en este tipo de data set donde es el seguimiento de un cliente que compran en linea y toma datos como el tiempo es muy poco probable que haya datos repetidos o que haya registros con los mismos datos en todas las columnas, dando a entender que en este caso los repetidos si son repetidos.

In [37]:
# Conteo de Duplicados
duplicados = df.duplicated().sum()
print("Registros duplicados:", duplicados)

Registros duplicados: 125


In [38]:
nulos = df.isna().sum().sort_values(ascending=False)
nulos[nulos > 0]

,0


In [39]:
# Trafic Type
df["TrafficType"].value_counts()

,count
TrafficType,
2,3913
1,2451
3,2052
4,1069
13,738
10,450
6,444
8,343
5,260


In [40]:
# Browser
df["Browser"].value_counts()

,count
Browser,
2,7961
1,2462
4,736
5,467
6,174
10,163
8,135
3,105
13,61


In [41]:
# Moths
df["Month"].value_counts()

,count
Month,
May,3364
Nov,2998
Mar,1907
Dec,1727
Oct,549
Sep,448
Aug,433
Jul,432
June,288


In [42]:
# Region
df["Region"].value_counts()

,count
Region,
1,4780
3,2403
4,1182
2,1136
6,805
7,761
9,511
8,434
5,318


In [43]:
# Operating System
df["OperatingSystems"].value_counts()

,count
OperatingSystems,
2,6601
1,2585
3,2555
4,478
8,79
6,19
7,7
5,6


In [44]:
# Special Days
df["SpecialDay"].value_counts()

,count
SpecialDay,
0.00,11079
0.60,351
0.80,325
0.40,243
0.20,178
1.00,154


In [45]:
# Visitor Type
df["VisitorType"].value_counts()

,count
VisitorType,
Returning_Visitor,10551
New_Visitor,1694
Other,85


In [46]:
# Weekend
df["Weekend"].value_counts()

,count
Weekend,
False,9462
True,2868


In [47]:
# Revenue
df["Revenue"].value_counts()

,count
Revenue,
False,10422
True,1908


## Limpieza de los Datos

In [48]:
# Copia del Dataframe para limpiar los datos
df_copy = df.copy()


In [49]:
# Eliminar los duplicados
df_copy = df_copy.drop_duplicates()
# Cantidad de datos despues de eliminar duplicados
print("Registros duplicados:", df_copy.duplicated().sum())

Registros duplicados: 0


In [50]:
# Cantidad de registros
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12205 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12205 non-null  int64  
 1   Administrative_Duration  12205 non-null  float64
 2   Informational            12205 non-null  int64  
 3   Informational_Duration   12205 non-null  float64
 4   ProductRelated           12205 non-null  int64  
 5   ProductRelated_Duration  12205 non-null  float64
 6   BounceRates              12205 non-null  float64
 7   ExitRates                12205 non-null  float64
 8   PageValues               12205 non-null  float64
 9   SpecialDay               12205 non-null  float64
 10  Month                    12205 non-null  object 
 11  OperatingSystems         12205 non-null  int64  
 12  Browser                  12205 non-null  int64  
 13  Region                   12205 non-null  int64  
 14  TrafficType              12

In [51]:
# Crear Variables auxiliares
# Segundos a minutos
df_copy["Administrative_Duration_min"] = df_copy["Administrative_Duration"] / 60
df_copy["Informational_Duration_min"] = df_copy["Informational_Duration"] / 60
df_copy["ProductRelated_Duration_min"] = df_copy["ProductRelated_Duration"] / 60
# Tiempo total en segundos y minutos
df_copy["Total_Duration_sec"] = df_copy["Administrative_Duration"] + df_copy["Informational_Duration"] + df_copy["ProductRelated_Duration"]
df_copy["Total_Duration_min"] = df_copy["Total_Duration_sec"] / 60
# Páginas totales
df_copy["Total_Pages"] = df_copy["Administrative"] + df_copy["Informational"] + df_copy["ProductRelated"]
# Segmento según la cantidad de páginas visitadas
df_copy["navigation_segment"] = pd.qcut(
    df_copy["Total_Pages"],
    q=3,
    labels=["Baja", "Media", "Alta"]
)
# Muestras de las variables creadas
df_copy[["Total_Duration_sec", "Total_Duration_min", "Total_Pages", "navigation_segment"]].head(10)


,Total_Duration_sec,Total_Duration_min,Total_Pages,navigation_segment
0,0.00,0.00,1,Baja
1,64.00,1.07,2,Baja
2,0.00,0.00,1,Baja
3,2.67,0.04,2,Baja
4,627.50,10.46,10,Baja
5,154.22,2.57,19,Media
6,0.00,0.00,1,Baja
7,0.00,0.00,1,Baja
8,37.00,0.62,2,Baja
9,738.00,12.30,3,Baja


In [52]:
# Porcentaje del tiempo registrado dedicado a páginas de productos
df_copy["ProductRelated_Time_pct"] = (
    df_copy["ProductRelated_Duration"]
    / df_copy["Total_Duration_sec"].replace(0, float("nan"))
    * 100
)

display(
    df_copy[
        ["ProductRelated_Duration", "Total_Duration_sec", "ProductRelated_Time_pct"]
    ].head(10)
)

,ProductRelated_Duration,Total_Duration_sec,ProductRelated_Time_pct
0,0.00,0.00,NaN
1,64.00,64.00,100.00
2,0.00,0.00,NaN
3,2.67,2.67,100.00
4,627.50,627.50,100.00
5,154.22,154.22,100.00
6,0.00,0.00,NaN
7,0.00,0.00,NaN
8,37.00,37.00,100.00
9,738.00,738.00,100.00


## Exploración

### Crosstab Analysis

In [53]:
# Corsstab entre month y revenue
crosstab_month_revenue = pd.crosstab(df_copy["Month"], df_copy["Revenue"])
crosstab_month_revenue

Revenue,False,True
Month,,
Aug,357,76
Dec,1490,216
Feb,178,3
Jul,366,66
June,256,29
Mar,1668,192
May,2964,365
Nov,2222,760
Oct,434,115


In [54]:
# Corsstab entre weekend y revenue
crosstab_weekend_revenue = pd.crosstab(df_copy["Weekend"], df_copy["Revenue"])
crosstab_weekend_revenue

Revenue,False,True
Weekend,,
False,7937,1409
True,2360,499


In [55]:
# Corsstab entre visitor type y revenue
crosstab_visitor_type_revenue = pd.crosstab(df_copy["VisitorType"], df_copy["Revenue"])
crosstab_visitor_type_revenue

Revenue,False,True
VisitorType,,
New_Visitor,1271,422
Other,65,16
Returning_Visitor,8961,1470


#Patrones Básicos

## Frecuencias Absolutas y Relativas

## Perfiles y Comparación de Grupos

## Tablas Cruzadas

# Construcción de Segmentos

## Creación de Categorías

## Segmentación por Reglas

## Visualización de Patrones

## Comportamientos Atípicos

# Segmentación Automática

## Similitud

##Distancia Euclidiana

## Escalas

## Esandarización

## K-Means

## Método del Codo

## Interpretación de Clusters